In [1]:
using LensFactory
using LensFactory.Constants
using LensFactory.LensModel.LensModelIO
using JLD2
using Interpolations
using CairoMakie

include("FreeFormLens.jl")
Threads.nthreads()

1

In [ ]:
function make_gridfrom_model(model::LensModel.ModelConfig)
    """
    Constructs a grid based on the model configuration.
    Returns the grid coordinates (gridx, gridy).
    """
    X_max, Y_max = model.observation.FOV
    pixel_scale = model.observation.pixel_scale
    gridx, gridy = Lenses.get_meshgrid(X_max, Y_max, pixel_scale)

    return gridx, gridy
end

function refine_map(map_coarse::M, gridx::M, gridy::M, resolution::T) where {T <:RV, M <: ROA}
    """
    Uses Bicubic interpolation to return a higher res map of the convergence κ.
    """
    x_nodes = range(gridx[1, 1], stop=gridx[end, 1], length=size(gridx, 2))
    y_nodes = range(gridy[1, 1], stop=gridy[1, end], length=size(gridy, 1))

    itp = interpolate(map_coarse, BSpline(Cubic(Line(OnGrid()))))
    itp = scale(itp, x_nodes, y_nodes)

    x_fine, y_fine = Lenses.get_meshgrid(maximum(x_nodes), maximum(y_nodes), resolution)

    map_fine = itp.(x_fine, y_fine)


    println("before returning")
    println(size(map_fine))
    println(x_fine)
    println(y_fine)

    return map_fine, x_fine, y_fine

end

function predict_image(lens::Lenses.AbstractLens, gridx::M, gridy::M, θx::N, θy::N, adis::T) where {T <: RV, M <: ROA, N <: ROA}
    """
    Predicts the image positions based on the lens model and source positions.
    """
    αx, αy = Lenses.get_deflection(lens, θx, θy)

    βx = θx .- αx .* adis
    βy = θy .- αy .* adis

    μ_obs = Lenses.get_magnification_image(lens, θx, θy, adis)

    # Calculate barycenter source position
    βx_model = sum(βx .* μ_obs.^2) / sum(μ_obs.^2)
    βy_model = sum(βy .* μ_obs.^2) / sum(μ_obs.^2)

    images = Lenses.get_image(lens, gridx, gridy, adis, (βx_model, βy_model))
    fig,axes = Lenses.plot_image_plane(lens, gridx, gridy, adis, source = (βx_model, βy_model))
    display(fig)
    return images

end

function give_image_rmsscatter(model::LensModel.ModelConfig, lens::Lenses.AbstractLens, param_ref::Dict{Tuple{Symbol,Symbol},Float64}, gridx::M, gridy::M) where M <: ROA
    """
    Computes the image positions based on the model configuration and lens parameters.
    Returns a scatter plot of the image positions.
    """
    adis = LensModel.LensModelUtils.adis_current(model, param_ref)

    sid = 1
    sum_rms = 0.0
    count = 0
    for src in model.source_config.sources
        adis_value = adis[sid]

        for knot in src.knots
            x = knot.x
            y = knot.y

            images = predict_image(lens, gridx, gridy, x, y, adis_value)
            println("images size: ", size(images))
            println("x size: ", size(x))
            println("y size: ", size(y))
            pred_imagesx = first.(images)
            pred_imagesy = last.(images)
            println("pred_imagesx size: ", size(pred_imagesx))
            println("pred_imagesy size: ", size(pred_imagesy))
            sum_rms += sum((pred_imagesx .- x).^2 .+ (pred_imagesy .- y).^2)
            count += length(x)
        end
        sid += 1
    end
    
    return sqrt(sum_rms / count)
end

function give_inversehessian(κ::M) where M <: ROA
    """
    Returns the covariance matrix (inverse Hessian) of the convergence map κ. This assumes a 
    gaussian distribution near the minima of the target function.
    """
    global prior_kappa, gridx, gridy, model, param_ref
    κ_vec = vec(κ)
    hessian = zeros(length(κ_vec), length(κ_vec))

    f0 = neg_logpost_MEM(κ_vec)
    h = 1e-5

    Threads.@threads for i in eachindex(κ_vec)
        buf = copy(κ_vec)
        buf[i] += h
        f1 = neg_logpost_MEM(buf)

        for j in eachindex(κ_vec)
            buf2 = copy(buf)
            buf2[j] += h
            f2 = neg_logpost_MEM(buf2)

            buf3 = copy(κ_vec)
            buf3[j] += h
            f3 = neg_logpost_MEM(buf3)

            hessian[i, j] = (f2 - f1 - f3 + f0) / (h^2)
            println("Hessian computation progress: ", i, ",", j)
        end
    end    
    return inv(hessian)
end

function give_errormap(hessian::M) where M <: ROA
    """
    Computes the error map from the inverse Hessian matrix.
    """
    global gridx
    diag_elements = diag(hessian)
    errormap = reshape(sqrt.(diag_elements), size(gridx))
    return errormap
end

give_errormap (generic function with 1 method)

In [ ]:
name = "MEM_fit_result8x8_1_125FOV"
filename = "../Diagnostics/files/$name" * ".jld2"

global prior_kappa, gridx, gridy, model, param_ref, reg_factor
# loading the jld2 file
data = load(filename)
model = data["model_config"]
κ_map = data["κ_map"]
prior_kappa = data["prior_kappa"]
reg_factor = data["reg_factor"]
errors = data["errors"]
param_ref = Dict(p.key => p.refer for p in model.parameters)

# making the grid
gridx, gridy = make_gridfrom_model(model)

res = 1  # 1 arcsec resolution for the refined map
κ_fine, x_fine, y_fine = refine_map(κ_map, gridx, gridy, res)
# initialize the lens
free_lens = FreeFormLens.init_FreeFormLens(κ_fine, x_fine, y_fine)
println("Lens initialized.")

UndefVarError: UndefVarError: `load` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
zs = 9
zd = model.observation.z_d
cosmo = Cosmology.init_cosmology()      # default cosmo
Dds = Cosmology.angular_diameter_distance(cosmo, zd, zs)
Ds = Cosmology.angular_diameter_distance(cosmo, 0.0, zs)
adis = Dds / Ds

0.7949451375771017

In [ ]:
err_fig, err_axes = Lenses.plot_sky(gridx, gridy)
hm = heatmap!(err_axes, gridx[:,1], gridy[1,:], errors, colormap = :turbo, colorrange = (0, maximum(errors)))
cb = Colorbar(err_fig[1,2], hm; label = "δκ", width = 20)
save("../Diagnostics/plots/$name" * "_error_map.png", err_fig)

rel_errors = errors ./ κ_map
relerr_fig, relerr_axes = Lenses.plot_sky(gridx, gridy)
hm_rel = heatmap!(relerr_axes, gridx[:,1], gridy[1,:], rel_errors, colormap = :turbo, colorrange = (0, 2))
cb_rel = Colorbar(relerr_fig[1,2], hm_rel; label = "δκ/κ", width = 20)
save("../Diagnostics/plots/$name" * "_relative_error_map.png", relerr_fig)


In [21]:
μ_fig, μ_axes = Lenses.plot_magnification_map(free_lens, x_fine, y_fine, adis, heatmap_kws = (colormap = :turbo, colorrange = (0,100)))
save("../Diagnostics/plots/$name" * "_magnification_map.png", μ_fig)

In [15]:
cc_fig, cc_axes = Lenses.plot_image_plane(free_lens, x_fine, y_fine, adis, two_panel = true)
save("../Diagnostics/plots/$name" * "_critical_curves.png", cc_fig)

In [11]:
κ_fig, κ_axes = Lenses.plot_surface_density(free_lens, x_fine, y_fine, adis, unit = :convergence, heatmap_kws = (colormap = :turbo, colorrange = (0,maximum(κ_fine))))
save("../Diagnostics/plots/$name" * "_kappa_map.png", κ_fig)

InterruptException: InterruptException:

In [17]:
prof_fig, prof_axis = Lenses.plot_magnification_profile(free_lens, x_fine, y_fine, adis)
save("../Diagnostics/plots/$name" * "_magnification_profile.png", prof_fig)

In [ ]:
println("χ² of predicted image positions: ", data["chi2"])

InterruptException: InterruptException: